In [ ]:

# ============================================================
# SECTION 1: IMPORTS AND CONFIGURATION
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image

# ------------------------------------------------------------
# Reproducibility and device
# ------------------------------------------------------------

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "preprocessed"

MODEL_PATH = PROJECT_ROOT / "models" / "baseline_unet_best.pth"

RESULTS_DIR = PROJECT_ROOT / "results" / "evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Model checkpoint:", MODEL_PATH)
print("Results:", RESULTS_DIR)

assert DATA_DIR.exists(), "Dataset directory not found"
assert MODEL_PATH.exists(), "Model checkpoint not found"

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

IMAGE_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = 0
THRESHOLD = 0.5

print("\nEvaluation configuration loaded.")


In [ ]:

# ============================================================
# SECTION 2: DATASET AND TEST DATALOADER
# ============================================================

class BUSIDataset(Dataset):

    def __init__(self, split_dir):
        self.image_dir = split_dir / "images"
        self.mask_dir = split_dir / "masks"

        self.image_paths = sorted(self.image_dir.glob("*.png"))

        assert len(self.image_paths) > 0, f"No images found in {self.image_dir}"

        for image_path in self.image_paths:
            mask_path = self.mask_dir / image_path.name
            assert mask_path.exists(), f"Missing mask: {mask_path}"

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]
        mask_path = self.mask_dir / image_path.name

        image = Image.open(image_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.float32)

        mask = (mask > 0).astype(np.float32)

        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


# ------------------------------------------------------------
# Create test dataset and loader
# ------------------------------------------------------------

test_dataset = BUSIDataset(DATA_DIR / "test")

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Test images:", len(test_dataset))
print("Test batches:", len(test_loader))

# Verify one batch
images, masks = next(iter(test_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)
print("Image range:", images.min().item(), "to", images.max().item())
print("Mask values:", torch.unique(masks))


In [ ]:

# ============================================================
# SECTION 3: LIGHTWEIGHT U-NET ARCHITECTURE
# Exact architecture used during training
# ============================================================

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        # Bottleneck
        self.bottleneck = DoubleConv(128, 256)

        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2)

        # Decoder
        self.up3 = nn.ConvTranspose2d(
            256, 128, kernel_size=2, stride=2
        )
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(
            64, 32, kernel_size=2, stride=2
        )
        self.dec1 = DoubleConv(64, 32)

        # Output
        self.output = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


# Initialize model
model = UNet(in_channels=1, out_channels=1).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())

print("U-Net initialized successfully.")
print("Device:", DEVICE)
print("Total trainable parameters:", f"{total_params:,}")

with torch.no_grad():
    sample_output = model(images.to(DEVICE))

print("Input shape:", images.shape)
print("Output shape:", sample_output.shape)
print("Output device:", sample_output.device)

assert total_params == 1_926_433, "Architecture parameter mismatch!"


In [ ]:

# ============================================================
# SECTION 4: LOAD TRAINED BASELINE CHECKPOINT
# ============================================================

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

# Load trained weights
model.load_state_dict(checkpoint["model_state_dict"])

# Evaluation mode
model.eval()

print("Baseline checkpoint loaded successfully.")

print("Best epoch:", checkpoint["epoch"])
print("Best validation Dice:", checkpoint["best_val_dice"])

print("Model parameters:", f"{total_params:,}")
print("Model device:", next(model.parameters()).device)

# Verify model output after loading trained weights
with torch.no_grad():
    sample_output = model(images.to(DEVICE))

print("Prediction shape:", sample_output.shape)
print("Checkpoint verification complete.")


In [ ]:

# ============================================================
# SECTION 5: EVALUATION METRICS
# ============================================================

def calculate_image_metrics(prediction, target, smooth=1e-7):

    prediction = prediction.bool()
    target = target.bool()

    intersection = (prediction & target).sum().item()

    pred_pixels = prediction.sum().item()
    target_pixels = target.sum().item()

    union = (prediction | target).sum().item()

    dice = (2.0 * intersection + smooth) / (
        pred_pixels + target_pixels + smooth
    )

    iou = (intersection + smooth) / (union + smooth)

    tp = intersection
    fp = (prediction & ~target).sum().item()
    fn = (~prediction & target).sum().item()

    precision = tp / (tp + fp + smooth)
    sensitivity = tp / (tp + fn + smooth)

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "sensitivity": sensitivity,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "predicted_pixels": pred_pixels,
        "target_pixels": target_pixels
    }


# ------------------------------------------------------------
# Quick metric function verification
# ------------------------------------------------------------

sample_target = masks[0, 0].numpy() > 0
sample_prediction = sample_target.copy()

verification = calculate_image_metrics(
    sample_prediction,
    sample_target
)

print("Metric function verification:")
print("Dice:", verification["dice"])
print("IoU:", verification["iou"])
print("Precision:", verification["precision"])
print("Sensitivity:", verification["sensitivity"])


In [ ]:

# ============================================================
# SECTION 5: EVALUATION METRICS
# ============================================================

def calculate_image_metrics(prediction, target, smooth=1e-7):

    # Convert inputs to NumPy boolean arrays
    prediction = np.asarray(prediction).astype(bool)
    target = np.asarray(target).astype(bool)

    intersection = np.logical_and(prediction, target).sum()

    pred_pixels = prediction.sum()
    target_pixels = target.sum()

    union = np.logical_or(prediction, target).sum()

    dice = (2.0 * intersection + smooth) / (
        pred_pixels + target_pixels + smooth
    )

    iou = (intersection + smooth) / (union + smooth)

    tp = intersection
    fp = np.logical_and(prediction, ~target).sum()
    fn = np.logical_and(~prediction, target).sum()

    precision = tp / (tp + fp + smooth)
    sensitivity = tp / (tp + fn + smooth)

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "sensitivity": sensitivity,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "predicted_pixels": int(pred_pixels),
        "target_pixels": int(target_pixels)
    }


# ------------------------------------------------------------
# Verify metric function
# ------------------------------------------------------------

sample_target = masks[0, 0].numpy() > 0
sample_prediction = sample_target.copy()

verification = calculate_image_metrics(
    sample_prediction,
    sample_target
)

print("Metric function verification:")
print("Dice:", verification["dice"])
print("IoU:", verification["iou"])
print("Precision:", verification["precision"])
print("Sensitivity:", verification["sensitivity"])


In [ ]:

# ============================================================
# SECTION 6: BASELINE TEST SET EVALUATION
# ============================================================

test_results = []

model.eval()

with torch.no_grad():

    for batch_idx, (images, masks) in enumerate(test_loader):

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(images)
        probabilities = torch.sigmoid(logits)

        predictions = probabilities >= THRESHOLD

        for i in range(images.size(0)):

            prediction = predictions[i, 0].cpu().numpy()
            target = masks[i, 0].cpu().numpy()

            metrics = calculate_image_metrics(
                prediction,
                target
            )

            is_normal = metrics["target_pixels"] == 0

            test_results.append({
                "image_name": test_dataset.image_paths[
                    batch_idx * BATCH_SIZE + i
                ].name,
                "is_normal": is_normal,
                **metrics
            })


# Convert results into DataFrame
test_results_df = pd.DataFrame(test_results)

# Separate lesion and normal images
lesion_df = test_results_df[
    test_results_df["is_normal"] == False
].copy()

normal_df = test_results_df[
    test_results_df["is_normal"] == True
].copy()


# ------------------------------------------------------------
# Lesion segmentation metrics
# ------------------------------------------------------------

print("========== BASELINE TEST RESULTS ==========")

print("\nLesion segmentation:")
print("Lesion images:", len(lesion_df))
print(f"Mean Dice: {lesion_df['dice'].mean():.4f}")
print(f"Mean IoU: {lesion_df['iou'].mean():.4f}")

print("\nPer-image precision and sensitivity:")
print(f"Mean Precision: {lesion_df['precision'].mean():.4f}")
print(f"Mean Sensitivity: {lesion_df['sensitivity'].mean():.4f}")


# ------------------------------------------------------------
# Global pixel-level metrics across all images
# ------------------------------------------------------------

total_tp = test_results_df["tp"].sum()
total_fp = test_results_df["fp"].sum()
total_fn = test_results_df["fn"].sum()

global_precision = total_tp / (total_tp + total_fp + 1e-7)
global_sensitivity = total_tp / (total_tp + total_fn + 1e-7)

print("\nGlobal pixel-level metrics:")
print(f"Precision: {global_precision:.4f}")
print(f"Sensitivity: {global_sensitivity:.4f}")


# ------------------------------------------------------------
# Normal image evaluation
# ------------------------------------------------------------

false_positive_normals = (
    normal_df["predicted_pixels"] > 0
).sum()

normal_fp_rate = (
    false_positive_normals / len(normal_df)
    if len(normal_df) > 0 else 0
)

print("\nNormal image evaluation:")
print("Normal images:", len(normal_df))
print("False-positive normal images:", false_positive_normals)
print(f"Normal false-positive rate: {normal_fp_rate:.4f}")


# ------------------------------------------------------------
# Save detailed results
# ------------------------------------------------------------

output_path = RESULTS_DIR / "baseline_test_results.csv"

test_results_df.to_csv(output_path, index=False)

print("\nDetailed results saved:", output_path)


In [ ]:

# ============================================================
# SECTION 7: INSPECT DATASET METADATA
# ============================================================

manifest_path = PROJECT_ROOT / "data" / "processed" / "preprocessed_manifest.csv"
splits_path = PROJECT_ROOT / "data" / "processed" / "dataset_splits.csv"

manifest_df = pd.read_csv(manifest_path)
splits_df = pd.read_csv(splits_path)

print("MANIFEST COLUMNS:")
print(manifest_df.columns.tolist())

print("\nSPLIT COLUMNS:")
print(splits_df.columns.tolist())

print("\nMANIFEST SAMPLE:")
display(manifest_df.head())

print("\nSPLIT SAMPLE:")
display(splits_df.head())


In [ ]:

# ============================================================
# SECTION 7A: VERIFY TEST IMAGE CATEGORY MAPPING
# ============================================================

print("Manifest sample:")
display(manifest_df[["image_id", "category", "split"]].head(5))

print("\nSplit metadata sample:")
display(splits_df[["filename", "category", "split"]].head(5))

print("\nTest prediction filenames:")
display(test_results_df[["image_name", "is_normal"]].head(5))

print("\nTest categories:")
print(
    splits_df[splits_df["split"] == "test"]["category"]
    .value_counts()
)


In [ ]:

# ============================================================
# SECTION 7B: CHECK FILENAME CORRESPONDENCE
# ============================================================

print("Manifest test samples:")
display(
    manifest_df[manifest_df["split"] == "test"][
        ["image_id", "category", "image_path"]
    ].head(10)
)

print("\nSplit metadata test samples:")
display(
    splits_df[splits_df["split"] == "test"][
        ["filename", "category", "image_path"]
    ].head(10)
)

print("\nPrediction filenames:")
print(test_results_df["image_name"].head(10).tolist())


In [ ]:

# ============================================================
# SECTION 7C: MAP PREDICTIONS TO CATEGORIES
# ============================================================

# Create a lookup table from the original split metadata
test_metadata = splits_df[
    splits_df["split"] == "test"
][["filename", "category"]].copy()

# Normalize prediction filenames
test_results_df["filename"] = (
    test_results_df["image_name"]
    .str.replace(
        r"^(benign|malignant|normal)_",
        "",
        regex=True
    )
)

# Merge category labels
test_results_df = test_results_df.merge(
    test_metadata,
    on="filename",
    how="left",
    validate="one_to_one"
)

# Verify mapping
print("Category mapping:")
print(test_results_df["category"].value_counts(dropna=False))

print("\nUnmatched images:",
      test_results_df["category"].isna().sum())

assert test_results_df["category"].notna().all(), (
    "Some images could not be mapped to categories."
)

# ------------------------------------------------------------
# Category-wise lesion evaluation
# ------------------------------------------------------------

print("\n========== CATEGORY-WISE RESULTS ==========")

for category in ["benign", "malignant"]:

    category_df = test_results_df[
        test_results_df["category"] == category
    ]

    print(f"\n{category.upper()}")
    print("Images:", len(category_df))
    print(f"Mean Dice: {category_df['dice'].mean():.4f}")
    print(f"Mean IoU: {category_df['iou'].mean():.4f}")
    print(f"Mean Precision: {category_df['precision'].mean():.4f}")
    print(f"Mean Sensitivity: {category_df['sensitivity'].mean():.4f}")


# Save updated detailed results
output_path = RESULTS_DIR / "baseline_test_results.csv"
test_results_df.to_csv(output_path, index=False)

print("\nUpdated results saved:", output_path)


In [ ]:

# ============================================================
# SECTION 8: SAVE BASELINE SUMMARY
# ============================================================

summary_rows = []

for category in ["benign", "malignant", "normal"]:

    category_df = test_results_df[
        test_results_df["category"] == category
    ]

    if category == "normal":

        false_positive_count = (
            category_df["predicted_pixels"] > 0
        ).sum()

        summary_rows.append({
            "category": category,
            "images": len(category_df),
            "mean_dice": np.nan,
            "mean_iou": np.nan,
            "mean_precision": np.nan,
            "mean_sensitivity": np.nan,
            "false_positive_images": false_positive_count,
            "false_positive_rate": (
                false_positive_count / len(category_df)
            )
        })

    else:

        summary_rows.append({
            "category": category,
            "images": len(category_df),
            "mean_dice": category_df["dice"].mean(),
            "mean_iou": category_df["iou"].mean(),
            "mean_precision": category_df["precision"].mean(),
            "mean_sensitivity": category_df["sensitivity"].mean(),
            "false_positive_images": np.nan,
            "false_positive_rate": np.nan
        })


baseline_summary = pd.DataFrame(summary_rows)

display(baseline_summary)

summary_path = RESULTS_DIR / "baseline_summary.csv"

baseline_summary.to_csv(summary_path, index=False)

print("\nBaseline summary saved:", summary_path)
